# 02 — Feature Store: member features ingested to SageMaker Feature Store

**Team Update 3.1 steps 4–5.** Ingests the temporally-split training table
(`aai-540-g1/processed/training/training_table.csv`, built by
`src/build_training_data.py`) into a SageMaker Feature Store feature group,
then proves the offline-store → Athena dataset-extraction path that the
training notebook consumes.

| Step | Where | Verify (📸) |
|---|---|---|
| Feature group create + ingest | this notebook | `get_record` spot-checks |
| Offline store sync | this notebook | Athena split-count query |

**Run order:** top to bottom in SageMaker Studio. If the SDK was already
imported before cell 2 ran, restart the kernel (Kernel → Restart) and re-run.
The offline store syncs asynchronously — the final query may need a re-run
after a few minutes (exactly like Lab 3.1).

## 1. Setup

Pin the SDK: v3 removed the Feature Store APIs this notebook uses (the
`FeatureDefinitions, value: 0` and `sagemaker.session` errors).

In [1]:
%pip install -q "sagemaker<3"

import boto3
import sagemaker
from sagemaker.session import Session

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client("sagemaker", region_name=region)
featurestore_runtime = boto_session.client("sagemaker-featurestore-runtime", region_name=region)

sess = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)
try:
    role = sagemaker.get_execution_role()  # Studio / notebook instance
except Exception:
    # local run (personal account): dedicated project role if present,
    # otherwise the newest SageMaker execution role in the account
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        if not cands:
            raise RuntimeError("no SageMaker execution role found - run scripts/ensure_execution_role.py")
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"
print(f"role={role}")
bucket = sess.default_bucket()
prefix = "aai-540-g1"

print(f"role={role}\nbucket={bucket}\nregion={region}")

Note: you may need to restart the kernel to use updated packages.


sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/sourangshupal/Library/Application Support/sagemaker/config.yaml


/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Couldn't call 'get_role' to get Role ARN from role name final-usd to get Role path.


/Users/sourangshupal/Downloads/final-project-usd/.venv-sm/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


role=arn:aws:iam::685057748560:role/aai-540-g1-execution-role


role=arn:aws:iam::685057748560:role/aai-540-g1-execution-role
bucket=sagemaker-us-east-1-685057748560
region=us-east-1


## 2. Load the training table from S3

In [2]:
import pandas as pd

boto3.client("s3").download_file(
    bucket, f"{prefix}/processed/training/training_table.csv", "training_table.csv"
)
df = pd.read_csv("training_table.csv")

# Feature Store record identifiers are strings; event_time must be a numeric
# epoch (the tz-aware datetime dtype fails load_feature_definitions - Lab 3.1).
df["person_id"] = df["person_id"].astype(str)
df["event_time"] = pd.Timestamp.now(tz="UTC").timestamp()

COLUMN_ORDER = [c for c in df.columns if c != "event_time"] + ["event_time"]
df = df[COLUMN_ORDER]
print(f"{len(df):,} rows x {df.shape[1]} cols")
df.head(3)

82,624 rows x 26 cols


,person_id,age,age_band,gender,race,state,observation_months,inpatient_visits,outpatient_visits,total_visits,...,renal_disease,drug_exposures,distinct_drugs,drug_eras,distinct_conditions,distinct_procedures,risk_class,split,label_rule_version,event_time
0,34,89.0,85+,Female,1.0,NY,11,0,6,13,...,0,24,24,26,22,20,Medium,train,outcome_score_v1: score = total_visits + 3*inp...,1.791319e+09
1,41,76.0,75-84,Male,1.0,TX,11,0,7,21,...,0,43,43,48,32,30,High,production,outcome_score_v1: score = total_visits + 3*inp...,1.791319e+09
2,83,45.0,<65,Female,1.0,GA,11,2,11,56,...,1,53,51,56,104,83,High,train,outcome_score_v1: score = total_visits + 3*inp...,1.791319e+09


## 3. Create the feature group and ingest

One row per member: 20 history-window features + `risk_class` (label) +
`split` (train/validation/test/production) + `event_time`.

In [3]:
import time
from sagemaker.feature_store.feature_group import FeatureGroup

FEATURE_GROUP_NAME = "aai-540-g1-member-features"

# Idempotent create: a previous (possibly partial) run's group is deleted
# first - re-ingesting over an existing group would leave duplicate
# event_time rows in the offline store and skew the Athena row counts.
try:
    sagemaker_client.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
    print("deleted previous feature group (partial run)")
    time.sleep(20)
except sagemaker_client.exceptions.ResourceNotFound:
    pass

feature_group = FeatureGroup(name=FEATURE_GROUP_NAME, sagemaker_session=sess)
feature_group.load_feature_definitions(data_frame=df)  # required before create

feature_group.create(
    s3_uri=f"s3://{bucket}/{prefix}/features",
    record_identifier_name="person_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

status = feature_group.describe().get("FeatureGroupStatus")
while status == "Creating":
    print("Waiting for Feature Group Creation")
    time.sleep(5)
    status = feature_group.describe().get("FeatureGroupStatus")
assert status == "Created", f"unexpected status: {status}"
print(f"FeatureGroup {FEATURE_GROUP_NAME} created.")

deleted previous feature group (partial run)


Waiting for Feature Group Creation


Waiting for Feature Group Creation


Waiting for Feature Group Creation


FeatureGroup aai-540-g1-member-features created.


In [4]:
# Bulk ingestion: the runtime API only offers single-record PutRecord
# (batch_put_record does not exist - verified against botocore), so we
# parallelize client-side with a thread pool. The SDK's own ingest() walks
# records serially and took >1h for 82k rows against online-store TPS limits;
# 16 workers finish in minutes. Throttled calls are retried with backoff.
import time
from concurrent.futures import ThreadPoolExecutor
from botocore.exceptions import ClientError

records = df.astype(object).where(pd.notna(df), None).to_dict("records")
t0 = time.time()


def put_one(rec):
    payload = {
        "FeatureGroupName": FEATURE_GROUP_NAME,
        "Record": [{"FeatureName": k, "ValueAsString": str(v) if v is not None else "0"}
                   for k, v in rec.items()],
    }
    delay = 1.0
    for _ in range(6):
        try:
            featurestore_runtime.put_record(**payload)
            return None
        except ClientError as exc:
            if exc.response["Error"]["Code"] in ("ThrottlingException", "ServiceUnavailable"):
                time.sleep(delay)
                delay *= 2
                continue
            raise
    return rec  # give up after 6 backoffs; counted below


done = failed = 0
with ThreadPoolExecutor(max_workers=16) as pool:
    for result in pool.map(put_one, records, chunksize=64):
        done += 1
        failed += result is not None
        if done % 10000 == 0:
            print(f"{done:,}/{len(records):,} records ({time.time() - t0:.0f}s)")
print(f"Ingested {done - failed:,} of {len(records):,} member records "
      f"in {time.time() - t0:.0f}s ({failed} failures)")

10,000/82,624 records (148s)


20,000/82,624 records (296s)


30,000/82,624 records (443s)


40,000/82,624 records (591s)


50,000/82,624 records (738s)


60,000/82,624 records (884s)


70,000/82,624 records (1037s)


80,000/82,624 records (1188s)


Ingested 82,624 of 82,624 member records in 1227s (0 failures)


## 4. 📸 Online store spot-checks

`get_record` for three members — one from the train split, one from
production, one from test (verify against the printed rows).

In [5]:
for split_name in ["train", "production", "test"]:
    pid = df.loc[df["split"] == split_name, "person_id"].iloc[0]
    record = featurestore_runtime.get_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierValueAsString=str(pid),
    )
    feats = {f["FeatureName"]: f["ValueAsString"] for f in record.get("Record", [])}
    expected = df[df["person_id"] == str(pid)].iloc[0]
    ok = feats.get("risk_class") == expected["risk_class"] and \
         feats.get("split") == split_name and \
         int(float(feats.get("total_visits", -1))) == int(expected["total_visits"])
    print(f"person {pid} [{split_name}] risk={feats.get('risk_class')} "
          f"visits={feats.get('total_visits')} -> {'MATCH' if ok else 'MISMATCH'}")

person 34 [train] risk=Medium visits=13 -> MATCH


person 41 [production] risk=High visits=21 -> MATCH


person 1103 [test] risk=Medium visits=0 -> MATCH


## 5. 📸 Offline store dataset extraction (Athena)

The offline store lands asynchronously (Kinesis Firehose buffer — usually
5–15 minutes). Re-run the count query until `n_total` equals 82,624, then
screenshot: it is the proof that Feature Store → training-dataset extraction
works for all four splits.

In [6]:
athena = boto_session.client("athena", region_name=region)
WORKGROUP = "aai540_g1"

def run_query(sql):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=WORKGROUP)[
        "QueryExecutionId"]
    while True:
        state = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if state["State"] == "SUCCEEDED":
            return athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
        if state["State"] in ("FAILED", "CANCELLED"):
            raise RuntimeError(state.get("StateChangeReason"))
        time.sleep(2)

# Offline-store Glue table: <feature-group-name>-<creation-epoch>. The offline
# store syncs asynchronously (Kinesis Firehose, typically 5-15 min after
# ingestion) - if the table or data is not there yet, say so instead of
# failing the notebook; re-run this cell later.
try:
    tables = boto_session.client("glue", region_name=region).get_tables(
        DatabaseName="sagemaker_featurestore")["TableList"]
    fg_table = next(t["Name"] for t in tables if FEATURE_GROUP_NAME in t["Name"])
    rows = run_query(
        f'SELECT split, risk_class, COUNT(*) AS n FROM "sagemaker_featurestore"."{fg_table}" '
        f"GROUP BY split, risk_class ORDER BY split, risk_class")
    for row in rows:
        print(" | ".join(c.get("VarCharValue", "") for c in row["Data"]))
    total = run_query(f'SELECT COUNT(*) AS n FROM "sagemaker_featurestore"."{fg_table}"')
    print("n_total:", total[1]["Data"][0].get("VarCharValue"), "(target 82,624)")
except Exception as exc:
    print(f"offline store not queryable yet ({str(exc)[:160]}). It syncs via "
          f"Kinesis Firehose 5-15 min after ingestion - re-run this cell.")

offline store not queryable yet (). It syncs via Kinesis Firehose 5-15 min after ingestion - re-run this cell.


## Optional cleanup (after grading)

Feature Store charges accrue while the online store exists. After the project
is graded, delete the feature group:

```python
sagemaker_client.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
```